# Título

**Autores:** Artur Lacerda Pugoni, Guilherme da Cunha Fonseca, Sarah Félix Stocler  
**Repositório do Projeto:** [\[GitHub\]](https://github.com/r3lc0ts/Projeto-AEDV)

---

## 1. Introdução:
Texto

## 2. Descrição e Preparação dos Dados
Os dados foram retirados Portal Brasileiro de Dados Abertos (Ministério do Turismo / Polícia Federal). 
* **Nota Metodológica:** Os registros refletem o **estado de entrada (UF de desembarque/fronteira)** no Brasil, e não necessariamente o destino final do turista. Além disso, os anos de **2020, 2021 e 2022** apresentam registros zerados na base original em decorrência da pandemia e serão tratados separadamente para não distorcer as análises de tendência histórica.

In [ ]:
# Carregamento e limpeza da base de dados 

import pandas as pd
import numpy as np
import unicodedata
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
# Procura no caminho todos os arquivos que começam com chegadas e terminam com .csv e coloca em ordem cronológica
arquivos = sorted(Path("../data/raw").glob("chegadas_*.csv"))
print(f"Total de arquivos encontrados: {len(arquivos)}")
# Converte o nome do mês em número para organizar e padroniar entre as tabelas de cada ano
MESES = {
    "janeiro": 1, "fevereiro": 2, "marco": 3, "abril": 4, "maio": 5, "junho": 6,
    "julho": 7, "agosto": 8, "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12
}
# Função para limpar o texto (snake_case)
def limpar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).strip().lower())
    return texto.encode("ascii", "ignore").decode().replace(" ", "_")

def carregar(arquivo):
    # Tenta ler em utf-8 primeiro para não quebrar acentos nos anos recentes; senão usa latin-1
    # lê o arquivo inteiro de uma vez e só então decidir o tipo de cada coluna
    try:
        df = pd.read_csv(arquivo, sep=";", encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(arquivo, sep=";", encoding="latin-1", low_memory=False)
        
    df.columns = [limpar(c) for c in df.columns]
    # Remover colunas inúteis
    df = df.drop(columns=[c for c in df.columns if c.startswith(("ordem", "cod"))])
    df = df.rename(columns={"via_de_acesso": "via"})

    # Para cada coluna de texto converte para string e remove espaços extras no começo e no fim
    for col in ["continente", "pais", "uf", "via"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    # .map(): pega cada valor da coluna passa por uma função e devolve o resultado
    # lambda m: def converter(m)
    # busca a chave. Se achar, devolve o valor. Se não achar, devolve o segundo argumento
    df["mes"] = df["mes"].map(lambda m: MESES.get(limpar(m), pd.to_numeric(m, errors="coerce")))
    # se não for número altera para NaN
    df["chegadas"] = pd.to_numeric(df["chegadas"], errors="coerce")
    return df
# dicionário onde a chave é o nome do arquivo sem extensão
# empilha todas as tabelas em uma só
tabelas = {arquivo.stem: carregar(arquivo) for arquivo in arquivos}
chegadas = pd.concat(tabelas.values(), ignore_index=True)

# Padroniza chegadas
chegadas["via"] = chegadas["via"].replace({"Aéreo": "Aérea", "Marítimo": "Marítima"})
# Agrupa linhas iguais, elimina duplicatas da coluna chegada, se todos os valores forem NaN mantém NaN e não 0
chaves = ["continente", "pais", "uf", "via", "ano", "mes"]
chegadas = (
    chegadas
    .groupby(chaves, as_index=False, dropna=False)["chegadas"]
    .sum(min_count=1)
)

# Base sem os anos de 2020, 2021 e 2022
chegadas_analise = chegadas[~chegadas["ano"].isin([2020, 2021, 2022])].copy()

# Salva a base limpa em data/processed
Path("../data/processed").mkdir(parents=True, exist_ok=True)
chegadas_analise.to_csv("../data/processed/chegadas_limpo.csv", index=False, encoding="utf-8")

print(f"Base consolidada com sucesso! Total de linhas: {len(chegadas_analise):,}")
display(chegadas_analise.head())

Total de arquivos encontrados: 0


ValueError: No objects to concatenate